---
title: Week 4.2, Real world Application, Simulation of Travertine formation
subtitle: Orchestra Scenario, Lorah & Herman
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-30
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

In [1]:
# Import libraries required for running all simulations
import locale
# locale.setlocale(locale.LC_ALL, 'en_US.UTF-8') # for formatting numbers with commas
locale.setlocale(locale.LC_ALL, 'C') # 

import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

# Prepare a file to capture PyOrchestra output
capture_file = open("pyorchestra_output.log", "w")


# pyOrchestra is implemented in C++
# Save original stdout file descriptor
# original_stdout_fd = sys.stdout.fileno()

# Duplicate original stdout so we can restore it later
# saved_stdout_fd = os.dup(original_stdout_fd)



# We need to import some Orchestra files. We need to know the path layout on 
# the local machine:
def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:


# Example usage:
# input_file = path_from_book_root("content", "week 02", "Orchestra_simulation", "chemistry1.inp")
# print("Input file:", input_file)

# orchestra_path = path_from_book_root("content", "project_THe", "Travertine_Example_Tutorial","Travertine_Orchestra")
# print(orchestra_path)

orchestra_path = '.'

## Unravelling the geochemistry of Travertine deposition along a stream
The data for this exercise are taken from:
**Reference:** Lorah, M.M. & Herman J.S. (1988) – *The chemical evolution of a travertine-depositing stream: Geochemical processes and mass transfer reactions*  
<https://doi-org.tudelft.idm.oclc.org/10.1029/WR024i009p01541>  

In the paper by Lorah and Herman (1988) you will see that several sampling trips were made to collect samples along a stream moving away from the hot springs where the stream originates, past a waterfall and then further down stream. In Table 1, some analysis results are presented for to sampling campaigns, one on 14 October 1984 and another on 6  April 1985. We will use these data for our analysis.

The aim of the analysis is to increase our understanding of the formation of travertine deposits along the stream and especially near the water fall. The analysis is similar to described in the paper.

The analysis consists in principle of two main steps:
1. We use the analysis results from the table to assess the the most likely in-situ conditions at the moment of sampling. The analysis results, give totals of the elements in mg/L. The true speciation will vary, elements will be present in other forms as well, solid minerals may control the composition of the water, but are not sampled with the water. The system may, or may not be in equilibrium. We analyze this problem with Orchestra using an Chemistry input file where we give the total of all measured compounds as master species, and we do not allow any minerals to precipitate, but we do report the SI-values for those minerals.
2. After doing the first analysis, we have additional information compared to the chemical analysis results from the lab, we know partial $\text{CO}_2\text{[g]}$ pressure and the SI-values for possible minerals for the water samples along the river. It is safe to assume that water down stream from the spring originated from the spring and that its composition should be related to the water at the spring. With this assumption we can use Orchestra to simulate changes related to the measurements obtained above. For example: if we have water from S-1 and move this to the conditions at F-6 ($\text{CO}_2[g]$ pressure and Calcite-SI), how much calcite would have precipitated in between?

```{figure} ./GIS/map_for_report.png
:label: Map from paper
:align: center
:width: 100%
Figure 1: Map from the paper projected on OpenStreetMap.
```

In Orchestra you define the type of calculation you want to do with a *chemistry.inp* file. For this assignment we need two types:
1. To calculate the equilibrium calculations for our system without allowing minerals to precipitate;
2. As similar input file, but where we allow the minerals to precipitate. As we want to compare samples along the river, we also need to be able to control the SI-value which is used to achieve the equilibrium. Normally, the SI-value is assumed to be zero for a sample in equilibrium, however, in this assignment we need to allow the model to equilibrate assuming higher, or even lower SI-values. This we achieve by manually adding a constant to the equilibrium equation in the *chemistry.inp* file. In addition, the $\text{CO}_2\text{[g]}$ pressure needs to be fixed as well, which we can achieve by fixing the CO2[g].logact.

### Extracting the coordinates of the measurement points
Because we have georeferenced the map from the paper to the OpenStreetMap using QGIS we can extract the x and y coordinates from the map. In addition we used QGIS to calculate the distance along the stream from the source. We can import these data so that we can plot the calculated results along the distance. 


In [2]:
# Add extracted coordinates and length along Falling Spring Creek to a dataframe for later processing
coord_dict = {
    'S-1': [-79.9333752265005586, 37.86960346907188324,	0],
    'S-2': [-79.93378630600923884, 37.86936810963599953, 0.113],
    'S-3': [-79.93358754229075203, 37.86943586470219003, 0.061],
    'D-1': [-79.93435549302124343, 37.86930035450750154, 0.256],
    'D-2': [-79.93524992975443411, 37.86967835600746213, 0.484],
    'D-3': [-79.93543965875844037, 37.86937524175117176, 0.529],
    'F-1': [-79.94694933545359561, 37.86809462323325448, 3.585],
    'F-2': [-79.95186467914058426, 37.87099898234640705, 5.228],
    'F-3': [-79.94752328277034792, 37.8677809455992076,	 3.748],
    'F-4': [-79.94783233132551459, 37.86774609244633893, 3.823],
    'F-5': [-79.94981907203731453, 37.86894270792835471, 4.419],
    'F-6': [-79.95239447666375554, 37.87369412518480516, 5.954],
}

df_coord = pd.DataFrame(data=coord_dict, index =['x-coord', 'y_coord', 'distance']).T

df_coord

,x-coord,y_coord,distance
S-1,-79.933375,37.869603,0.000
S-2,-79.933786,37.869368,0.113
S-3,-79.933588,37.869436,0.061
D-1,-79.934355,37.869300,0.256
D-2,-79.935250,37.869678,0.484
D-3,-79.935440,37.869375,0.529
F-1,-79.946949,37.868095,3.585
F-2,-79.951865,37.870999,5.228
F-3,-79.947523,37.867781,3.748
F-4,-79.947832,37.867746,3.823


## Step 1: Assessing the water samples
For this first step we have prepared an Orchestra input file using the *orchestra2026.jar* Graphical User Interface (GUI). Please have a look at this file with the GUI. 
The approach was quite simple, we added all the analysed compounds that are found in the table as primary entities in the dissolved phase (*diss*). We define the pH as a fixed logact and we assume that water is the solute phase with a fixed logactivity of 0. We make sure that none of the mineral phases on the *Phases & Reactions* tab are selected, because we do not allow for any precipitation. We assume that the water samples are stored in bottles with out any gas-phase (head space), we set the gasvolume to be zero on the *Variables* tab. This chemical system is defined in *chemistry_Travertine.inp*.

In order to illustrate the steps, we will analyse two water samples: S1 and F1. Please note that the concentrations are given in $\text{mg/l}$ and the temperature in $^\text{o}\text{C}$, we need to translate these numbers to $\text{mol/l}$ and $\text{K}$ in order to work with Orchestra.

```{hint} Creating the *chemistry.inp* file
Using the Orchestra GUI shown in [](#master_species)  we are able to define a system of the primary entities. You do this by selecting the **Chemistry** tab on the right in the ORCHESTRA-COMPOSER window. The **Chemistry** tab shows a number of tabs which fill the *chemistry1.inp* file. We start by selecting tab **Primary entities/ Master Species**. Remember that the Master Species you select will be used to create all other species in the system using the mass-action laws. The interface in principle will not allow you to select to master species for the same component. The Master species are used to define the total moles present in the system (including the amounts of these master species present in the dervied dissolved, solids and gaseous species).
```


In [3]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
       ['mmass', 1e-3, 1e-3, 1e-3, 1.00794 + 12.0107 + 3*15.9994, 40.078, 24.305, 
         22.9898, 39.0983, 18.9984, 35.453, 32.065 + 4*15.9994],
       ['S-1', 24.3, 700, 7.32, 308, 165, 25.4, 5.1, 14.2,  1.0, 3.3, 283.],
       ['F-6', 14.4, 442, 8.25, 210, 122, 21.3, 3.3, 11.4, 0.7, 4, 246]
    ]
df_data_orig = pd.DataFrame(data=data,columns=headers)
df_data_orig.set_index('Sample',inplace=True)

# calculate molar concentrations
df_data = pd.DataFrame()
df_data = df_data_orig.loc[['S-1','F-6']]/df_data_orig.loc['mmass'] * 0.001
# calculate temperature in K
df_data['T'] = df_data['T_celsius'] + 273.15

# table_md_data = df_data.to_markdown(floatfmt='.4e')
# display(Markdown(table_md_data))
df_data

,T_celsius,Conductivity,pH,HCO3-.tot,Ca+2.tot,Mg+2.tot,Na+.tot,K+.tot,F-.tot,Cl-.tot,SO4-2.tot,T
Sample,,,,,,,,,,,,
S-1,24.3,700.0,7.32,0.005048,0.004117,0.001045,0.000222,0.000363,0.000053,0.000093,0.002946,297.45
F-6,14.4,442.0,8.25,0.003442,0.003044,0.000876,0.000144,0.000292,0.000037,0.000113,0.002561,287.55


In [4]:
# Define lists for inputs and outputs from pyOrchestra
inputs = [
    'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot', 
    'T', 'watervolume', 'gasvolume',
    'fixed_logact_CO2', 'deltaSIcalcite', 'deltaSIdolomite',
]

#outputs
outputs_logact = [
    'Alkalinity.logact','Ar.logact','Ar[g].logact','C.logact','CO2.logact','CO2[g].logact','CO3-2.logact',
    'C[+4].logact','Ca.logact','Ca+2.logact','CaCO3.logact','CaF+.logact','CaHCO3+.logact','CaHSO4+.logact',
    'CaOH+.logact','CaSO4.logact','Cl.logact','Cl-.logact','F.logact','F-.logact','H.logact','H+.logact',
    'H2CO3.logact','H2F2.logact','H2O.logact','H2O[g].logact','HCO3-.logact','HF.logact','HF2-.logact',
    'HSO4-.logact','H[+1].logact','K.logact','K+.logact','KSO4-.logact','Mg.logact','Mg+2.logact','MgCO3.logact',
    'MgF+.logact','MgHCO3+.logact','MgOH+.logact','MgSO4.logact','Na.logact','Na+.logact','NaCO3-.logact','NaF.logact',
    'NaHCO3.logact','NaSO4-.logact','O.logact','OH-.logact','O[-2].logact','S.logact','SO4-2.logact','S[+6].logact',
]

outputs_con = [
    'Alkalinity.con','Ar.con','Ar[g].con','C.con','CO2.con','CO2[g].con','CO3-2.con',
    'C[+4].con','Ca.con','Ca+2.con','CaCO3.con','CaF+.con','CaHCO3+.con','CaHSO4+.con',
    'CaOH+.con','CaSO4.con','Cl.con','Cl-.con','F.con','F-.con','H.con','H+.con',
    'H2CO3.con','H2F2.con','H2O.con','H2O[g].con','HCO3-.con','HF.con','HF2-.con',
    'HSO4-.con','H[+1].con','K.con','K+.con','KSO4-.con','Mg.con','Mg+2.con','MgCO3.con',
    'MgF+.con','MgHCO3+.con','MgOH+.con','MgSO4.con','Na.con','Na+.con','NaCO3-.con','NaF.con',
    'NaHCO3.con','NaSO4-.con','O.con','OH-.con','O[-2].con','S.con','SO4-2.con','S[+6].con',
]

outputs_min_si = [
    'Anhydrite[s].si','Aragonite[s].si','Artinite[s].si','Brucite[s].si','CO2g[s].si','Calcite[s].si',
    'Dolomite[d][s].si','Dolomite[s].si','Epsomite[s].si','Fluorite[s].si','Gypsum[s].si','Halite[s].si',
    'Huntite[s].si','Hydromagnesite[s].si', 'Magnesite[s].si','Mirabilite[s].si','Nahcolite[s].si',
    'Natron[s].si','Nesquehonite[s].si','Portlandite[s].si','Thenardite[s].si','Thermonatrite[s].si',
    'Trona[s].si',
]

outputs_min_tot = [
    'Anhydrite[s].tot','Aragonite[s].tot','Artinite[s].tot','Brucite[s].tot','CO2g[s].tot','Calcite[s].tot',
    'Dolomite[d][s].tot','Dolomite[s].tot','Epsomite[s].tot','Fluorite[s].tot','Gypsum[s].tot','Halite[s].tot',
    'Huntite[s].tot','Hydromagnesite[s].tot', 'Magnesite[s].tot','Mirabilite[s].tot','Nahcolite[s].tot',
    'Natron[s].tot','Nesquehonite[s].tot','Portlandite[s].tot','Thenardite[s].tot','Thermonatrite[s].tot',
    'Trona[s].tot',
]

outputs_extra = [
    'chargebalance', 'totcharge', 'pressure', 'I',
]

outputs_master_diss = [  
    'HCO3-.diss', 'Ca+2.diss', 'Mg+2.diss', 'Na+.diss', 'K+.diss', 'F-.diss', 'Cl-.diss', 'SO4-2.diss'
]

outlist = (outputs_logact + outputs_con +
        outputs_min_si + outputs_min_tot + outputs_master_diss +
        inputs + outputs_extra)


### Analysis of the equilibrium state of the original sample

In [5]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_Travertine.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array(inputs)
    
    # We select the output from Orchestra we need to use
    outlist = (outputs_logact + outputs_con +
        outputs_min_si + outputs_min_tot + outputs_master_diss +
        inputs + outputs_extra)

    OutVars1 = np.array(outlist)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)


Reading and expanding calculator new stylechemistry_Travertine.inp
Scanning file: chemistry_Travertine.inp
Scanning file: objects2026_THe.txt
Scanning file: chemistry_Travertine.inp
Scanning file: objects2026_THe.txt
Including file: chemistry_Travertine.inp
Scanning file: objects2026_THe.txt
0.025 sec.
	Reading variables .... 0.016 s
testing:
14:pH
16:HCO3-.tot
9:Ca+2.tot
20:Mg+2.tot
22:Na+.tot
18:K+.tot
13:F-.tot
11:Cl-.tot
24:SO4-2.tot
25:T
26:watervolume
27:gasvolume
28:fixed_logact_CO2
29:deltaSIcalcite
30:Alkalinity.logact
31:Ar.logact
7:Ar[g].logact
32:C.logact
33:CO2.logact
34:CO2[g].logact
35:CO3-2.logact
36:C[+4].logact
37:Ca.logact
8:Ca+2.logact
38:CaCO3.logact
39:CaF+.logact
40:CaHCO3+.logact
41:CaHSO4+.logact
42:CaOH+.logact
43:CaSO4.logact
44:Cl.logact
10:Cl-.logact
45:F.logact
12:F-.logact
46:H.logact
47:H+.logact
48:H2CO3.logact
49:H2F2.logact
2:H2O.logact
50:H2O[g].logact
15:HCO3-.logact
51:HF.logact
52:HF2-.logact
53:HSO4-.logact
54:H[+1].logact
55:K.logact
17:K+.logac

In [6]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples

# Initialize the pyOrchestra Input array can create a OrchInput data frame 
# for easy evaluation
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
OrchInput = pd.DataFrame(columns=InVars1, index=df_data.index)

# Initialize Res_Simulation dataframe to capture the pyOrchestra output using OutVars1
Res_Simulation = pd.DataFrame(columns=OutVars1, index=df_data.index)

# set default constants
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = 0.0 # not used for this calculation
IN1[0][np.where(InVars1 == 'deltaSIcalcite')] = 0.0 # not used for this calculation

# loop over available samples in df_data
for sample in df_data.index:
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman.

    # please note, we fill until -4, constants have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-4] = df_data.loc[sample, InVars1[:-4]].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)

    # Store input and output in dataframes
    OrchInput.loc[sample] = IN1[0]    
    Res_Simulation.loc[sample] = OUT[0]

# Print a summary of the output in order to do first check on the results
Res_List = [
    'pH','T', 'I','HCO3-.tot',
    'Ca+2.tot', 'HCO3-.con', 'Ca+2.con', 'HCO3-.diss', 'Ca+2.diss',
    'CO2[g].con','CO2[g].logact','Calcite[s].si','Dolomite[s].si',
    'Gypsum[s].si', 'Fluorite[s].si',
]

table_mdini = Res_Simulation[Res_List].to_markdown(floatfmt='.4e')

# display(Markdown(table_mdini))
Res_Simulation[Res_List]

,pH,T,I,HCO3-.tot,Ca+2.tot,HCO3-.con,Ca+2.con,HCO3-.diss,Ca+2.diss,CO2[g].con,CO2[g].logact,Calcite[s].si,Dolomite[s].si,Gypsum[s].si,Fluorite[s].si
Sample,,,,,,,,,,,,,,,
S-1,7.32,297.450012,0.014958,0.005048,0.004117,0.0033,0.003357,0.004095,0.004117,0.009143,-2.038901,0.242137,0.028155,-0.950475,-0.916443
F-6,8.25,287.549988,0.012518,0.003442,0.003044,0.003112,0.002509,0.00335,0.003044,0.000883,-3.053851,0.906454,1.266993,-1.06615,-1.256857


## Simulation to find out what the water composition could have been in the sub-surface
We revisit the analysis of sample S-1 and then run a scenario where we assume that minerals are present in large amounts. The analysis of S-1 indicates the present of calcium and magnesium carbonates, sulphates and flourites in the subsurface.

The most likely minerals associated with these species are:
| mineral | chemical formula |
| --- | --- |
| Calcite[s] | $\text{CaCO}_3\text{[s]}$ |
| Dolomite[s] | $\text{CaMg(CO}_3\text{)[s]}$ |
| Gypsum[s] | $ \text{CaSO}_4.6\text{H}_2\text{O[s]}$ |
| Fluorite[s] | $ \text{CaF}_2\text{[s]} $|

We can assume that the system contains sigificant amounts of these minerals. In addition the temperature is elevated and the partial pressure of $\text{CO}_2\text{[g]}$ is significantly higher than in the atmosphere. 

For our system, all minerals mentioned above are present in at least 5 moles / liter. The temperature is elevated to 60 $^o\text{C}$, and the partial pressure of $\text{CO}_2\text{[g]}$ is 3 atm. We need to add an excess amount of carbonate to allow the equilibrium with the partial CO2 pressure.

This implies the following initial amounts per liter for the master species:
| master species | moles |
| --- | --- |
| HCO3-.tot | 50 |
| Ca+2.tot  | 20 |
| Mg+2.tot  | 5 |
| Na+.tot   | 0.000222 |
| K+.tot    | 0.000363 |
| F-.tot    | 10 |
|Cl-.tot     | 0.000093 |
|SO4-2.tot |  5 |
| T        | 313.15 |

We start our simulation using the precipitating version from above. We change the initial amounts of master species, ensure that all minerals listed can precipitate and then run our simulation. To do this we first adapt the chemistry input file to the one in *chemistry_Travertine_underground.inp* where the above minerals are marked to precipitate.

In [7]:
print ('Analysed data for S-1: ', df_data.loc['S-1'])

Analysed data for S-1:  T_celsius        24.300000
Conductivity    700.000000
pH                7.320000
HCO3-.tot         0.005048
Ca+2.tot          0.004117
Mg+2.tot          0.001045
Na+.tot           0.000222
K+.tot            0.000363
F-.tot            0.000053
Cl-.tot           0.000093
SO4-2.tot         0.002946
T               297.450000
Name: S-1, dtype: float64


In [8]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry2.inp' file.
# Later we use pO2, InVars2 and OutVars2
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_Travertine_UG.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars3 = np.array(inputs)
    
    OutVars3 = np.array(outlist)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO3 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above','
    pO3.initialise(InputFile, NoCells, InVars3, OutVars3)

Try a first calculation with iia switched off....
Parsing expressions of chemistry_Travertine.inp..... 
Optimizing expressions of chemistry_Travertine.inp..... 0.094 sec.
3848 variables, 11090 expressions, 10 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
This was successful!!
Reading and expanding calculator new stylechemistry_Travertine_UG.inp
Scanning file: chemistry_Travertine_UG.inp
Scanning file: objects2026_THe.txt
Scanning file: chemistry_Travertine_UG.inp
Scanning file: objects2026_THe.txt
Including file: chemistry_Travertine_UG.inp
Scanning file: objects2026_THe.txt
0.027 sec.
	Reading variables .... 0.016 s
testing:
15:pH
17:HCO3-.tot
10:Ca+2.tot
21:Mg+2.tot
23:Na+.tot
19:K+.tot
14:F-.tot
12:Cl-.tot
25:SO4-2.tot
37:T
38:watervolume
39:gasvolume
40:fixed_logact_CO2
41:deltaSIcalcite
42:Alkalinity.logact
43:Ar.logact
8:Ar[g].logact
44:C.logact
45:CO2.logact
46:CO2[g].logact
47:CO3-2.logact
48:C[+4].logact
49:Ca.log

In [9]:
# Step 1, initial condition.
# InVars need to contain floats
IN3 = np.array([np.ones_like(InVars3)]).astype(float)
# OrchInput = pd.DataFrame(columns=InVars3, index=df_data.index)
# Initialize Res_Simulation dataframe to capture the pyOrchestra output using OutVars1
# Res_Simulation = pd.DataFrame(columns=OutVars3, index=df_data.index)

# set default watervolume and gasvolume
IN3[0][np.where(InVars3 == 'Ar[g].logact')] = -20 # per liter
IN3[0][np.where(InVars3 == 'watervolume')] = 1.0 # per liter
IN3[0][np.where(InVars3 == 'gasvolume')] = 0.0 # no gas 

# Set the initial amounts of the master species to sample S-1
IN3[0][:-5] = df_data.loc['S-1', InVars3[:-5]].values

# We add the masses of species to form the solid phases

IN3[0][np.where(InVars3 == 'HCO3-.tot')] = 50
IN3[0][np.where(InVars3 == 'Ca+2.tot')] = 20 + df_data.loc['S-1', 'Ca+2.tot']
IN3[0][np.where(InVars3 == 'Mg+2.tot')] = 5 + df_data.loc['S-1', 'Mg+2.tot']
IN3[0][np.where(InVars3 == 'F-.tot')] = 10 + df_data.loc['S-1', 'F-.tot']
IN3[0][np.where(InVars3 == 'SO4-2.tot')] = 5 + df_data.loc['S-1', 'SO4-2.tot']

IN3[0][np.where(InVars3 == 'T')] = 313.15
IN3[0][np.where(InVars3 == 'fixed_logact_CO2')] = 0.478
IN3[0][np.where(InVars3 == 'deltaSIcalcite')] = 0

# Run Orchestra simulation to equilibrate the sample from S-1 to F-6
OUT = pO3.set_and_calculate(IN3)

# Define a label for the output index
simindex = 'underground'

OrchInput.loc[simindex] = IN3[0]    
Res_Simulation.loc[simindex] = OUT[0]

# print(Res_Sim2[['T','pH','Ca+2.con', 'HCO3-.con', 'CO2[g].con', 'Calcite[s].si', 'Calcite[s].tot', 'CO2[g].logact']])

# table_mdini = Res_Simulation[Res_List].to_markdown(floatfmt='.4e')
# display(Markdown(table_mdini))

Res_Simulation.loc['underground']

Alkalinity.logact   -6.285664
Ar.logact               -35.0
Ar[g].logact            -15.0
C.logact            -6.285664
CO2.logact          -1.155404
                       ...   
deltaSIcalcite            0.0
chargebalance             0.0
totcharge            0.065724
pressure              3.07881
I                    0.078356
Name: underground, Length: 178, dtype: object

In [10]:
# List all minerals with SI-values > 0.5
sel_large_SI = Res_Simulation.loc['underground', outputs_min_si] > 0

# Display the results in a nice output format
selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
# display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

Res_Simulation[selected_minerals]

,CO2g[s].si,Calcite[s].si,Dolomite[s].si
Sample,,,
S-1,-2.038901,0.242137,0.028155
F-6,-3.053851,0.906454,1.266993
underground,0.0,0.0,0.0


In [11]:
# List all minerals with SI-values > 0.5
sel_large_SI = Res_Simulation.loc['underground', outputs_min_tot] > 0

# Display the results in a nice output format
selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
# display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

Res_Simulation[selected_minerals]

,CO2g[s].tot,Calcite[s].tot,Dolomite[s].tot,Fluorite[s].tot,Gypsum[s].tot
Sample,,,,,
S-1,0.0,0.0,0.0,0.0,0.0
F-6,0.0,0.0,0.0,0.0,0.0
underground,34.508499,5.006787,4.991329,4.999951,4.982078


In [14]:
Res_List = ['pressure', 'CO2[g].con', 'CO2g[s].tot', 'CO2[g].logact','Calcite[s].si','Dolomite[s].si']
Res_Simulation[Res_List]

,pressure,CO2[g].con,CO2g[s].tot,CO2[g].logact,Calcite[s].si,Dolomite[s].si
Sample,,,,,,
S-1,1.03905,0.009143,0.0,-2.038901,0.242137,0.028155
F-6,1.017092,0.000883,0.0,-3.053851,0.906454,1.266993
underground,3.07881,3.006076,34.508499,0.478,0.0,0.0


In [13]:
Res_Simulation[outputs_master_diss]

,HCO3-.diss,Ca+2.diss,Mg+2.diss,Na+.diss,K+.diss,F-.diss,Cl-.diss,SO4-2.diss
Sample,,,,,,,,
S-1,0.004095,0.004117,0.001045,0.000222,0.000363,0.000053,0.000093,0.002946
F-6,0.00335,0.003044,0.000876,0.000144,0.000292,0.000037,0.000113,0.002561
underground,0.165899,0.02397,0.009716,0.000222,0.000363,0.000151,0.000093,0.020868


### Move the sample back to the position of S-1 by equilibrating against the original S-1 analysis without precipitation

In [29]:
# Step 1, initial condition.
# InVars need to contain floats
IN3 = np.array([np.ones_like(InVars3)]).astype(float)
# OrchInput = pd.DataFrame(columns=InVars3, index=df_data.index)
# Initialize Res_Simulation dataframe to capture the pyOrchestra output using OutVars1
# Res_Simulation = pd.DataFrame(columns=OutVars3, index=df_data.index)

# set default watervolume and gasvolume
IN3[0][np.where(InVars3 == 'Ar[g].logact')] = -20 # per liter
IN3[0][np.where(InVars3 == 'watervolume')] = 1.0 # per liter
IN3[0][np.where(InVars3 == 'gasvolume')] = 0.0 # no gas 

# Set the initial amounts of the master species to calculated composition of underground (i.e. masterspecies)
# Please note, InVars,has suffix tot, we need to use suffix diss to select correct values
# Normalize names by removing suffixes
invars_base = np.char.replace(InVars3, '.tot', '')
outputs_base = np.char.replace(outputs_master_diss, '.diss', '')
# Find indices
indices = np.where(np.isin(invars_base, outputs_base))[0]

IN3[0][indices] = Res_Simulation.loc['underground', outputs_master_diss]

# We now ensure that the right boundary conditions apply
IN3[0][np.where(InVars3 == 'T')] = Res_Simulation.loc['S-1', 'T']
IN3[0][np.where(InVars3 == 'fixed_logact_CO2')] = Res_Simulation.loc['S-1', 'CO2[g].logact']
IN3[0][np.where(InVars3 == 'deltaSIcalcite')] = Res_Simulation.loc['S-1', 'Calcite[s].si']
IN3[0][np.where(InVars3 == 'deltaSIdolomite')] = Res_Simulation.loc['S-1', 'Dolomite[s].si']

# Run Orchestra simulation to equilibrate the sample from S-1 to F-6
OUT = pO3.set_and_calculate(IN3)

# Define a label for the output index
simindex = 'underground to S-1'

OrchInput.loc[simindex] = IN3[0]    
Res_Simulation.loc[simindex] = OUT[0]

# Overwrite 
Res_Simulation.loc[simindex, 'Calcite[s].si'] = Res_Simulation.loc['S-1', 'Calcite[s].si']
Res_Simulation.loc[simindex, 'Dolomite[s].si'] = Res_Simulation.loc['S-1', 'Dolomite[s].si']

Res_Simulation.loc[simindex,outputs_master_diss]

HCO3-.diss    0.003217
Ca+2.diss     0.015883
Mg+2.diss      0.00387
Na+.diss      0.000222
K+.diss       0.000363
F-.diss       0.000119
Cl-.diss      0.000093
SO4-2.diss    0.018643
Name: underground to S-1, dtype: object

In [30]:
OrchInput

,pH,HCO3-.tot,Ca+2.tot,Mg+2.tot,Na+.tot,K+.tot,F-.tot,Cl-.tot,SO4-2.tot,T,watervolume,gasvolume,fixed_logact_CO2,deltaSIcalcite
Sample,,,,,,,,,,,,,,
S-1,7.32,0.005048,0.004117,0.001045,0.000222,0.000363,0.000053,0.000093,0.002946,297.45,1.0,0.0,0.0,0.0
F-6,8.25,0.003442,0.003044,0.000876,0.000144,0.000292,0.000037,0.000113,0.002561,287.55,1.0,0.0,0.0,0.0
underground,7.32,50.0,20.004117,5.001045,0.000222,0.000363,10.000053,0.000093,5.002946,313.15,1.0,0.0,0.478,0.0
underground to S-1,1.0,0.165899,0.02397,0.009716,0.000222,0.000363,0.000151,0.000093,0.020868,297.450012,1.0,0.0,-2.038901,0.242137


In [31]:
# List all minerals with SI-values > 0.5
sel_large_SI = Res_Simulation.loc['underground to S-1', outputs_min_si] > 0

# Display the results in a nice output format
selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
# display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

Res_Simulation[selected_minerals]

,Aragonite[s].si,CO2g[s].si,Calcite[s].si,Dolomite[s].si
Sample,,,,
S-1,0.060474,-2.038901,0.242137,0.028155
F-6,0.700607,-3.053851,0.906454,1.266993
underground,-0.146447,0.0,0.0,0.0
underground to S-1,0.047988,0.0,0.242137,0.028155


In [32]:
Res_List = ['pressure', 'CO2[g].con', 'CO2g[s].tot', 'CO2[g].logact','Calcite[s].si','Dolomite[s].si']
Res_Simulation[Res_List]

,pressure,CO2[g].con,CO2g[s].tot,CO2[g].logact,Calcite[s].si,Dolomite[s].si
Sample,,,,,,
S-1,1.03905,0.009143,0.0,-2.038901,0.242137,0.028155
F-6,1.017092,0.000883,0.0,-3.053851,0.906454,1.266993
underground,3.07881,3.006076,34.508499,0.478,0.0,0.0
underground to S-1,0.039039,0.009143,0.150074,-2.038901,0.242137,0.028155
